# M7P1 DAP-Seq Analysis Scripted

In this lab, we are performing analysis of DAP-Seq data from the Paper: [Cistrome and Epicistrome Features Shape the Regulatory DNA Landscape](https://www.sciencedirect.com/science/article/pii/S0092867416304810?via%3Dihub), located at [GSE60141](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE60141). We will use scripting, similar to what will be used in the final project, to complete the first part of the pipeline: 

### DAP-Seq Analysis Pipeline (Using `module_7_dapseq`)

1. **Data Acquisition**
   - Download raw reads from SRA using `fasterq-dump`

2. **Quality Control**
   - Assess read quality with `fastqc`.
   - Trim adapters and filter low-quality reads with `fastp`.
   - Summarize QC reports using `multiqc`.

3. **Read Alignment**
   - Align reads to the reference genome with `bowtie2`.
   - Convert, sort, and index BAM files with `samtools`.
   - Optional: mark duplicates with `picard`.

4. **Peak Calling**
   - Identify binding sites using `macs2`.
   - Process and manipulate intervals with `bedtools` or `bedops`.

5. **Visualization**
   - Generate coverage tracks and heatmaps with `deeptools`.
   - Convert BEDGraph to BigWig for genome browsers using `ucsc-bedgraphtobigwig`.
   - Visualize peaks in IGV

6. **Motif Analysis**
   - Discover enriched motifs with `meme` and `homer`.



```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

## Understanding Protocols

In the published protocol for Dap-Seq, [Mapping genome-wide transcription-factor binding sites using DAP-seq](http://dx.doi.org/10.1038/nprot.2017.055), the data analysis protocol is short and open-ended: 

*[Excerpt omitted: Bartlett et al. (2017) Nature Protocols 12:1659-1672, [doi:10.1038/nprot.2017.055](https://doi.org/10.1038/nprot.2017.055).]*

### How can we start from here?

In cases like this, it is rather daunting to think about how you would design A protocol for your own lab or research. The wet lab experiment is well documented and could be reproduced easily even across organisms outside of Arabidopsis; data analysis, however, is left open to interpretation. In this case, it would be useful to search for git repositories that contain similar protocols for processing DAP-Seq or ChIP-Seq data.

![nf-core-chipseq_metro_map_grey.png](images/M7P1_DAP_Seq_Analysis_Scripted_ec1a33a2.png)

*Image: nf-core/chipseq metro map, nf-core community, MIT License.*

Nf-core has a rather detailed protocol for use in ChIP-Seq.


# Dataset Info
The data used this week is from [GSE60141](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE60141).

In this module, we will reproduce analyses for DAP-Seq focusing on the ARF5/MONOPTEROS transcription factor and related ARF family members from the published dataset GSE60143.

We will: 
1. Download raw sequencing reads (100 bp single-end Illumina data) for two biological replicates of ARF5 expressed from E. coli with GST-fusion tags (GSM1925826 and GSM1925827), along with the critical beads-only control (GSM1924998) that captures non-specific DNA binding background.
2. Perform quality control using FastQC and fastp, align the reads to the TAIR10 Arabidopsis reference genome using Bowtie2 (the same aligner used in the paper).
3. Call peaks with MACS2 using the beads-only control for background subtraction to identify high-confidence DNA binding sites—expecting to find 3,000-5,000 ARF5 peaks as reported in the original study.
4. Following peak calling, we will use MEME-ChIP to discover the characteristic TGTC repeat motif that O'Malley et al. showed is bound cooperatively by ARF5, with phased repeats occurring at 10-12 bp spacing (direct repeats), 15-18 bp spacing (everted repeats), and 15-16 bp spacing (inverted repeats), corresponding to binding sites positioned on the same face of the DNA helix or across helical turns.
5. Validate our results by examining the IAA5 promoter, which the paper identified as having 13 phased TGTC motifs with 60-fold enrichment in ARF5 DAP-seq data, representing one of the strongest cooperative binding events in the Arabidopsis cistrome.
6. Generate genome-wide coverage tracks using deepTools for visualization in IGV or UCSC Genome Browser, and perform Gene Ontology enrichment analysis on ARF5 target genes, which should reveal significant enrichment for auxin response (p < 10⁻⁶⁰) and embryonic/root development processes consistent with ARF5's known role as MONOPTEROS.


![image.png](images/M7P1_DAP_Seq_Analysis_Scripted_fd6bf4c1.png)

## Sample Information Table

| Sample Type | Gene/TF     | AGI Locus       | GSM ID      | Sample Name                  | SRA Accession | Expression System      | Library Type   | Organism     | Notes                             |
|------------|-------------|----------------|------------|------------------------------|---------------|----------------------|----------------|-------------|-----------------------------------|
| Control    | Empty Vector | -              | GSM1924999 | control_tnt-pixhalo_colamp_v3a | SRX1411979    | TNT (HaloTag only)   | amp DAP        | A. thaliana | Primary control, methylation-free |
| Control    | Beads Only   | -              | GSM1924998 | control_none.beads_col_v3a   | SRX1411978    | None                 | Standard DAP   | A. thaliana | Beads background control          |
| Treatment  | ARF5/MP      | AT1G19850      | GSM1925826 | ARF_ecoli.MP_col_v31         | SRX1412806    | E. coli GST-fusion   | Standard DAP   | A. thaliana | Replicate 1, with methylation    |
| Treatment  | ARF2         | AT5G62000      | GSM1925138 | ARF_tnt.ARF2_col_v31         | SRX1412118    | TNT wheat germ HaloTag | Standard DAP | A. thaliana | Auxin response repressor          |
| Treatment  | ARF16        | AT4G30080      | GSM1925137 | ARF_tnt.ARF16_col_b          | SRX1412117    | TNT wheat germ HaloTag | Standard DAP | A. thaliana | Root cap development              |


## Create project folder and `module_7_dapseq` environment - 0_m7_project.sh

First, we need to create our project folder and environment, and retrieve our samples. Let's do this using some pre-made bash scripts and talk about how they work and why we chose to do this:

The first bash script we will utilize is `0_m7_project.sh`. This script creates the project directory and a Mamba environment that contains programs for use in DAP-Seq analysis. 

**The file tree for our project folder looks like this:**
```
m7_project/
├── data/
│   ├── raw/          # Downloaded FASTQ files
│   ├── trimmed/      # Quality-trimmed reads
│   └── sra/          # SRA prefetch cache
├── results/
│   ├── alignment/    # BAM files
│   ├── peaks/        # MACS2 output
│   ├── motifs/       # MEME/HOMER output
│   ├── qc/           # FastQC/MultiQC reports
│   └── bigwig/       # Genome browser tracks
├── references/
│   ├── genome/       # TAIR10 FASTA
│   ├── annotation/   # GFF3 files
│   └── bowtie2_index/ # Bowtie2 index files
├── scripts/          # Analysis scripts
└── logs/             # Job logs
```

The environment `module_7_dapseq` contains the following programs:
| Program                   | Purpose / Role in DAP-Seq Analysis                                                                 |
|----------------------------|---------------------------------------------------------------------------------------------------|
| sra-tools                 | Download sequencing data from NCBI SRA.                                                          |
| entrez-direct             | Access and download NCBI databases programmatically.                                             |
| fastqc                    | Perform quality control checks on raw sequencing reads.                                           |
| fastp                     | Trim adapters, filter low-quality reads, and generate QC reports.                                 |
| multiqc                   | Aggregate QC reports from FastQC, Fastp, and other tools into a single summary.                   |
| bowtie2                   | Align sequencing reads to a reference genome (used for DAP-seq read mapping).                     |
| samtools                  | Manipulate SAM/BAM files, including sorting, indexing, and filtering.                             |
| bamtools                  | Additional BAM file operations, such as merging and extracting subsets.                           |
| picard                    | Process BAM files for duplicate marking, metrics calculation, and format conversion.              |
| htslib                     | Library for reading/writing high-throughput sequencing data (BAM/CRAM/VCF).                       |
| tabix                     | Index and query compressed genomic data files (VCF, BED, etc.).                                   |
| bedtools                  | Intersect, merge, and manipulate genomic interval files (BED, GFF, etc.).                        |
| bedops                     | Efficient set operations on genomic intervals (merge, subtract, intersect).                       |
| macs2                     | Call peaks from aligned reads to identify transcription factor binding sites.                     |
| deeptools                  | Generate coverage tracks, heatmaps, and other visualizations of aligned reads.                   |
| ucsc-bedgraphtobigwig      | Convert BEDGraph files to BigWig format for genome browser visualization.                         |
| meme                      | Discover and analyze motifs enriched in peak regions.                                             |
| homer                     | Perform motif discovery and annotation in genomic intervals.                                      |
| pigz                      | Parallel compression of files to speed up large file handling.                                    |


To run this file, navigate to your module 7 project folder within the command line using a command like: 

`cd $WORK_DIR/module_7_dap_seq/`

Then, run the script using the `bash` command: 

`bash $REPO_DIR/pipelines/dap_seq/0_m7_project.sh`  (the scripts are in [`pipelines/dap_seq/`](../../../pipelines/dap_seq/README.md))

## Download samples from their GSE Accessions using `1_SRA_fetch.sh`

**MAKE SURE YOU HAVE AT LEAST 8 CPU CORES ON YOUR SESSION**

Previously, we did not know the sample accession names, and parsed the sample run info retrieved with `entriez-direct`. In this case, we know our samples and loaded them into a bash array within the bash script `1_SRA_fetch.sh`. It looks like this: 
```bash
declare -A SAMPLES=(
    ["SRX1411979"]="GSM1924999_control_empty_vector"
    ["SRX1411978"]="GSM1924998_control_beads"
    ["SRX1412806"]="GSM1925826_ARF5_rep1"
    ["SRX1412118"]="GSM1925138_ARF2"
    ["SRX1412117"]="GSM1925137_ARF16"
)
```

We then download the SRA files in parallel directly to `../m7_project/data/sra/` and convert them to `.fastq.gz` in the script. 

To run the script within the terminal, type: 

`cd $WORK_DIR/module_7_dap_seq/`

Then, run the script using the `bash` command: 

`bash 1_SRA_fetch.sh`

# Run QC, Trimming, and Alignment with Bowtie2 using `2_qc_alignment.sh`

In this step, we perform the following:

- Initial QC: FastQC on raw reads
- Adapter Trimming: fastp with quality filtering
- Post-trim QC: FastQC on trimmed reads
- Index Check: Build Bowtie2 index using TAIR10 genome
- Alignment: Bowtie2 generates sorted BAM files
- Statistics: Alignment rates and summary table
- MultiQC Report: Consolidated HTML report

To run the script within the terminal, type: 

`cd $WORK_DIR/module_7_dap_seq/`

Then, run the script using the `bash` command: 

`bash 2_qc_alignment.sh`

# Call peaks with `MACS2` using `3_macs2.sh`

In this step:
- Mark Duplicates: Picard MarkDuplicates (keeps all reads for MACS2)
- Duplication Stats: Calculate library complexity
- Peak Calling: MACS2 with beads-only control
- Peak QC: Count peaks, calculate FRiP, quality metrics
- Replicate Comparison: ARF5 rep1 vs rep2 overlap
- IGV Prep: Sort, compress, index for visualization

To run the script within the terminal, type: 

`cd $WORK_DIR/module_7_dap_seq/`

Then, run the script using the `bash` command: 

`bash 3_macs2.sh`

# Prepare files for visualization in IGV with `4_viz_prep.sh`

In this step: 

- Create chromosome sizes: This is required for BigWig conversion.  
- Generate BigWig from BAM: Produce coverage tracks with RPKM normalization.  
- Convert BedGraph to BigWig: Generate MACS2 signal tracks.  
- Create heatmaps: Visualize signal at peak regions within ±2 kb.  
- Generate profile plots: Calculate average signal across peaks.  
- Compare ARF5 replicates: Perform correlation analysis.  
- Create IGV session: Load all tracks at once for visualization.


To run the script within the terminal, type: 

`cd $WORK_DIR/module_7_dap_seq/`

Then, run the script using the `bash` command: 

`bash 4_viz_prep.sh`

# Use `HOMER` for motif discovery by running `5_motif.sh`

Now, we will: 

- Extract peak sequences - Select the top 500 peaks and include ±100 bp around the summits.  
-  Perform MEME-ChIP analysis - Conduct de novo motif discovery.  
-  Perform HOMER analysis - Identify both known and de novo motifs.  
-  Compare ARF5 replicates - Assess motif consistency using TOMTOM.  
-  Perform enrichment analysis - Calculate motif statistics.  
-  Scan for known ARF motifs - Search for TGTC/TGTCGG motifs using FIMO.

To run the script within the terminal, type: 

`cd $WORK_DIR/module_7_dap_seq/`

Then, run the script using the `bash` command: 

`bash 5_motif.sh`

# Annotate peaks using the `gtf` annotation of the TAIR10 Genome

Finally, prior to functional analysis, we will: 
- download the tair10 genome annotation
- annotate peaks
- categorize peaks by regions (promoter, genic, intergenic)
- Compare our targets by observing the overlap between ARF family members

To run the script within the terminal, type: 

`cd $WORK_DIR/module_7_dap_seq/`

Then, run the script using the `bash` command: 

`bash 6_annotate_peaks.sh`
